## Trabajo Práctico: ETL con Python y Pandas

A partir del archivo `ventas.json`, realiza un proceso ETL en Python utilizando Pandas como herramienta principal en sus tres fases. Responde las preguntas propuestas en **celdas Markdown**.

### Fase 1: Extracción

1. Carga `ventas.json` en un DataFrame de Pandas.
    - Presenta el total de registros, los nombres de los campos y los tipos de datos iniciales detectados por Pandas.

2. Realiza un perfil inicial del conjunto de datos utilizando `isna()`, `describe()` y `nunique()`. El perfil debe permitir observar:
    - la cantidad de valores ausentes por campo;
    - un resumen descriptivo de los datos;
    - la cantidad de valores únicos por campo.

A partir de los resultados obtenidos, responde en una celda Markdown: ¿qué problemas podríamos encontrar en el conjunto de datos y qué campos requerirían transformaciones o validaciones?

In [1]:
import pandas as pd

df_ventas = pd.read_json("ventas.json")

df_ventas

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,Formosa,25000,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,Resistencia,15000,5.0,enviado,2026-08-02,NaN
2,3,Notebook,Formosa,25000,2.0,entregado,2026-08-20,NaN
3,4,Notebook,Resistencia,25000,5.0,enviado,2026-08-08,NaN
4,5,Monitor,Resistencia,35000,1.0,pendiente,2026-08-23,NaN
...,...,...,...,...,...,...,...,...
1998,1999,Notebook,Resistencia,15000,4.0,entregado,2026-08-06,NaN
1999,2000,Notebook,formosa,35000,2.0,pendiente,2026-08-08,NaN
2000,1257,Mouse,Formosa,850000,3.0,enviado,2026-08-23,NaN
2001,1565,Notebook,Formosa,120000,3.0,entregado,2026-08-24,NaN


In [2]:
df_ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 2003 entries, 0 to 2002
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  2003 non-null   int64  
 1   producto            2003 non-null   str    
 2   ciudad              2003 non-null   str    
 3   precio              2001 non-null   object 
 4   cantidad            2002 non-null   float64
 5   estado              2002 non-null   str    
 6   fecha               2002 non-null   str    
 7   comentario_interno  1 non-null      str    
dtypes: float64(1), int64(1), object(1), str(5)
memory usage: 125.3+ KB


In [3]:
df_ventas.isna().sum()

id                       0
producto                 0
ciudad                   0
precio                   2
cantidad                 1
estado                   1
fecha                    1
comentario_interno    2002
dtype: int64

In [4]:
df_ventas.describe()

,id,cantidad
count,2003.000000,2002.000000
mean,1000.592611,3.026474
std,577.402809,1.488184
min,1.000000,-10.000000
25%,500.500000,2.000000
50%,1001.000000,3.000000
75%,1500.500000,4.000000
max,2000.000000,5.000000


In [5]:
df_ventas.nunique()

id                    2000
producto                 5
ciudad                   5
precio                  13
cantidad                11
estado                   8
fecha                   36
comentario_interno       1
dtype: int64

## De acuerdo con los resultados obtenidos:

### Problemas que se pueden encontrar
Valores ausentes (NaN): Las columnas de precio, cantidad, estado y fecha contienen registros nulos. 
y la columnas comentario_interno tiene 2002 NaN de un total de 2003 filas 

### Campos que requerirían transformaciones o validaciones
- Ciudad: contiene espacios(por ej., " Formosa ") y además mezcla mayúsculas y minúsculas ("Formosa", "formosa")
- Precio: Aparece como tipo de datos object, además existen valores negativos y nulos
- Cantidad: Contiene valores negativos y nulos.
- Estado: Contiene valores nulos.
- Fecha: Aparece como tipo de dato str(texto). Hay fechas con formato distinto (YYYY/MM/DD, DD/MM/YYYY), fechas no válidas o inexistentes (ej. 2026-02-30, 2026-13-01, "fecha-invalida") y valores nulos.
- comentario_interno: parece ser un columna irrelevante, ya que casi todos los campos contienen valor nulo.

### Fase 2: Transformación

Realiza las siguientes transformaciones y validaciones sobre el DataFrame utilizando Pandas.

3. Algunas ciudades aparecen escritas con diferencias de mayúsculas, minúsculas o espacios adicionales. Normaliza sus valores para que cada ciudad tenga una única representación.

4. Convierte los precios numéricos almacenados como texto (por ejemplo, `"25000"`) a un tipo numérico adecuado dentro del DataFrame.
    - Identifica como inválidas las ventas cuyos precios no puedan convertirse a un valor numérico.

5. ¿Qué reglas de negocio puedes establecer para los campos `cantidad` y `precio`?
    - Define y valida esas reglas para identificar las ventas inválidas.
    - **Pista**: Revisa el tipo de dato y el **rango** de los campos, con `.max()` y `.min()` de Pandas.

6. Valida mediante Pandas que el campo `estado` contenga uno de los siguientes valores:
    - `pendiente`
    - `enviado`
    - `entregado`

7. Valida que las fechas tengan el formato `YYYY-MM-DD` e identifica las ventas que no cumplan esta condición.

8. ¿Todos los campos son necesarios? Utiliza Pandas para excluir los campos innecesarios del conjunto de datos final.

9. Como última transformación, verifica si existen registros completamente duplicados. Informa cuántos encontraste y elimina las copias adicionales antes de construir el conjunto de datos final.

In [6]:
df_ventas_copy = df_ventas.copy()
df_ventas_copy['ciudad'] = df_ventas_copy["ciudad"].str.strip().str.upper()

df_ventas_copy['ciudad'].value_counts()

ciudad
FORMOSA        1202
RESISTENCIA     414
CLORINDA        387
Name: count, dtype: int64

In [7]:

df_ventas_copy['precio'] = pd.to_numeric(df_ventas_copy['precio'], "coerce")
precio_max = df_ventas_copy['precio'].max()
precio_min = df_ventas_copy['precio'].min()

print(f"Precio de venta máximo {precio_max}")
print(f"Precio de venta mínimo {precio_min}")
 #df_ventas_copy.info()

df_ventas_copy = df_ventas_copy[df_ventas_copy["cantidad"] > 0]
df_ventas_copy = df_ventas_copy[df_ventas_copy["precio"] > 0]

precio_max = df_ventas_copy['precio'].max()
precio_min = df_ventas_copy['precio'].min()

print(f"Precio de venta máximo {precio_max}")
print(f"Precio de venta mínimo {precio_min}")

Precio de venta máximo 850000.0
Precio de venta mínimo -25000.0
Precio de venta máximo 850000.0
Precio de venta mínimo 15000.0


In [8]:
df_ventas_copy["estado"] = df_ventas_copy["estado"].astype(str).str.strip().str.lower()
estados = ["pendiente", "enviado", "entregado"]
df_ventas_copy = df_ventas_copy[df_ventas_copy["estado"].isin(estados)]

df_ventas_copy

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,FORMOSA,25000.0,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,RESISTENCIA,15000.0,5.0,enviado,2026-08-02,NaN
2,3,Notebook,FORMOSA,25000.0,2.0,entregado,2026-08-20,NaN
3,4,Notebook,RESISTENCIA,25000.0,5.0,enviado,2026-08-08,NaN
4,5,Monitor,RESISTENCIA,35000.0,1.0,pendiente,2026-08-23,NaN
...,...,...,...,...,...,...,...,...
1998,1999,Notebook,RESISTENCIA,15000.0,4.0,entregado,2026-08-06,NaN
1999,2000,Notebook,FORMOSA,35000.0,2.0,pendiente,2026-08-08,NaN
2000,1257,Mouse,FORMOSA,850000.0,3.0,enviado,2026-08-23,NaN
2001,1565,Notebook,FORMOSA,120000.0,3.0,entregado,2026-08-24,NaN


In [9]:
df_ventas_copy["fecha"] = pd.to_datetime(df_ventas_copy["fecha"], format="%Y-%m-%d", errors="coerce")
df_ventas_copy = df_ventas_copy[df_ventas_copy["fecha"].notna()]

df_ventas_copy

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,FORMOSA,25000.0,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,RESISTENCIA,15000.0,5.0,enviado,2026-08-02,NaN
2,3,Notebook,FORMOSA,25000.0,2.0,entregado,2026-08-20,NaN
3,4,Notebook,RESISTENCIA,25000.0,5.0,enviado,2026-08-08,NaN
4,5,Monitor,RESISTENCIA,35000.0,1.0,pendiente,2026-08-23,NaN
...,...,...,...,...,...,...,...,...
1998,1999,Notebook,RESISTENCIA,15000.0,4.0,entregado,2026-08-06,NaN
1999,2000,Notebook,FORMOSA,35000.0,2.0,pendiente,2026-08-08,NaN
2000,1257,Mouse,FORMOSA,850000.0,3.0,enviado,2026-08-23,NaN
2001,1565,Notebook,FORMOSA,120000.0,3.0,entregado,2026-08-24,NaN


In [10]:
cant_duplicados = df_ventas_copy.duplicated().sum()
cant_duplicados

np.int64(3)

In [11]:
df_ventas_copy = df_ventas_copy.drop_duplicates()
df_ventas_copy

,id,producto,ciudad,precio,cantidad,estado,fecha,comentario_interno
0,1,Notebook,FORMOSA,25000.0,2.0,pendiente,2026-08-05,NaN
1,2,Notebook,RESISTENCIA,15000.0,5.0,enviado,2026-08-02,NaN
2,3,Notebook,FORMOSA,25000.0,2.0,entregado,2026-08-20,NaN
3,4,Notebook,RESISTENCIA,25000.0,5.0,enviado,2026-08-08,NaN
4,5,Monitor,RESISTENCIA,35000.0,1.0,pendiente,2026-08-23,NaN
...,...,...,...,...,...,...,...,...
1995,1996,Mouse,RESISTENCIA,120000.0,2.0,pendiente,2026-08-27,NaN
1996,1997,Mouse,FORMOSA,25000.0,1.0,entregado,2026-08-23,NaN
1997,1998,Mouse,CLORINDA,15000.0,4.0,enviado,2026-08-02,NaN
1998,1999,Notebook,RESISTENCIA,15000.0,4.0,entregado,2026-08-06,NaN


In [12]:
df_ventas_copy = df_ventas_copy.drop(columns=["comentario_interno"])

df_ventas_copy

,id,producto,ciudad,precio,cantidad,estado,fecha
0,1,Notebook,FORMOSA,25000.0,2.0,pendiente,2026-08-05
1,2,Notebook,RESISTENCIA,15000.0,5.0,enviado,2026-08-02
2,3,Notebook,FORMOSA,25000.0,2.0,entregado,2026-08-20
3,4,Notebook,RESISTENCIA,25000.0,5.0,enviado,2026-08-08
4,5,Monitor,RESISTENCIA,35000.0,1.0,pendiente,2026-08-23
...,...,...,...,...,...,...,...
1995,1996,Mouse,RESISTENCIA,120000.0,2.0,pendiente,2026-08-27
1996,1997,Mouse,FORMOSA,25000.0,1.0,entregado,2026-08-23
1997,1998,Mouse,CLORINDA,15000.0,4.0,enviado,2026-08-02
1998,1999,Notebook,RESISTENCIA,15000.0,4.0,entregado,2026-08-06


## Fase 3: Carga

A partir de los resultados obtenidos con Pandas:

10. Construye el DataFrame final de ventas **válidas y transformadas**. Toda venta que incumpla al menos una validación debe quedar excluida de este DataFrame.

11. Antes de guardar los resultados, presenta un resumen de calidad que incluya:

- cantidad total de ventas procesadas;
- cantidad de registros duplicados eliminados;
- cantidad de ventas válidas;
- cantidad de ventas rechazadas.

12. Guarda el DataFrame de ventas válidas en un nuevo archivo llamado `ventas_limpias.json`.

In [13]:
total_procesados = len(df_ventas)
cant_validas = len(df_ventas_copy)
cant_rechazadas = total_procesados - cant_validas
print("RESUMEN DE CALIDAD")
print(f"Cantidad total de ventas procesadas: {total_procesados}")
print(f"Cantidad de registros duplicados eliminados: {cant_duplicados}")
print(f"Cantidad de ventas válidas: {cant_validas}")
print(f"Cantidad de ventas rechazadas: {cant_rechazadas}")

df_ventas_copy.to_json("ventas_limpias.json", orient="records", indent=4)

RESUMEN DE CALIDAD
Cantidad total de ventas procesadas: 2003
Cantidad de registros duplicados eliminados: 3
Cantidad de ventas válidas: 1971
Cantidad de ventas rechazadas: 32


C:\Users\IPF-2026\AppData\Local\Temp\ipykernel_4140\678021714.py:10: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  df_ventas_copy.to_json("ventas_limpias.json", orient="records", indent=4)


In [14]:

nuevo_df = pd.read_json("ventas_limpias.json")

nuevo_df

,id,producto,ciudad,precio,cantidad,estado,fecha
0,1,Notebook,FORMOSA,25000,2,pendiente,1785888000000
1,2,Notebook,RESISTENCIA,15000,5,enviado,1785628800000
2,3,Notebook,FORMOSA,25000,2,entregado,1787184000000
3,4,Notebook,RESISTENCIA,25000,5,enviado,1786147200000
4,5,Monitor,RESISTENCIA,35000,1,pendiente,1787443200000
...,...,...,...,...,...,...,...
1966,1996,Mouse,RESISTENCIA,120000,2,pendiente,1787788800000
1967,1997,Mouse,FORMOSA,25000,1,entregado,1787443200000
1968,1998,Mouse,CLORINDA,15000,4,enviado,1785628800000
1969,1999,Notebook,RESISTENCIA,15000,4,entregado,1785974400000


## Desafío opcional: registro de *dead letters*

Además del proceso ETL principal, puedes conservar los errores detectados en un DataFrame separado. Cada fila debe representar un único error e incluir los datos originales completos de la venta, el campo inválido, su valor original y la razón del error. Si una venta presenta más de un error, debe generar una fila por cada validación incumplida.

Por ejemplo, una fila del DataFrame de errores podría tener la siguiente estructura:

```json
{
    "id": 101,
    "producto": "Mouse",
    "ciudad": "Formosa",
    "precio": -15000,
    "cantidad": 2,
    "estado": "pendiente",
    "fecha": "2026-08-12",
    "comentario_interno": "dato que no necesitamos",
    "campo_invalido": "precio",
    "valor_original": -15000,
    "razon": "el precio debe ser mayor que cero"
}
```

Amplía el resumen de calidad para incluir:

- cantidad total de errores;
- cantidad de errores por campo y por razón.

Explica en una celda Markdown por qué la cantidad de ventas rechazadas puede ser diferente de la cantidad total de errores. Finalmente, guarda el DataFrame de errores en un archivo llamado `errors.json`.

**Dato curioso:** en los procesos de datos, los registros que no pueden continuar por errores de validación suelen denominarse *dead letters*. Separarlos permite analizarlos y corregirlos sin detener el procesamiento de los registros válidos.

La cantidad de ventas rechazadas puede ser diferente de la cantidad total de erroes, porque en cantidad de ventas rechadas cuenta una sola operacion o fila como venta rechazada, mientras que la cantidad de errores va sumando de validaciones incumplidas dentro de una misma venta o fila rechazada